# 🧪 Benchmarking & Evaluasi Komparatif Model IndoBERT pada Dataset IndQNER-EL

**Proyek Skripsi:** Pengayaan Dataset *Indonesian Quranic Named Entity Recognition* (IndQNER) Berbasis *Entity Linking* (IndQEL)

## 📌 Parameter Resmi Sesuai `IndQNER/README.md`
Eksperimen ini mengadopsi konfigurasi hiperparameter yang digunakan pada paper resmi IndQNER (*Transfer Learning Setting*):
- **Pretrained Model**: `indobenchmark/indobert-base-p1`
- **Maximum Sequence Length**: `256`
- **Batch Size**: `16`
- **Learning Rate**: `2e-5`
- **Number of Epochs**: `10` (opsi: 20, 40)

### 🎯 Nilai Baseline Resmi dari Paper IndQNER (pada Epoch 10, Seq 256):
| Maximum Seq Length | Number of Epoch | Precision | Recall | F1-Score |
| :---: | :---: | :---: | :---: | :---: |
| 256 | 10 | 0.67 | 0.65 | 0.65 |

---

### 🔬 Desain Eksperimen Komparatif (Ablation Study):
1. **Model A (Baseline Model)**: Dilatih pada dataset asli (`IndQNER Asli`).
2. **Model B (Proposed Model)**: Dilatih pada dataset hasil revisi (`IndQNER-EL Revisi`).
3. **Evaluasi Adil**: Keduanya diuji pada **Test Set yang sama (`test.txt`)** menggunakan metrik standar industri NLP (**Strict Span Matching via `seqeval`**).

## 1. Setup Lingkungan & Instalasi Dependensi

In [6]:
# Periksa ketersediaan GPU (Disarankan GPU T4 di Google Colab)
!nvidia-smi

# Instalasi Hugging Face Transformers, Datasets, Seqeval, dan Accelerate
!pip install -q transformers datasets seqeval accelerate

Thu Oct 15 03:25:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Kloning Repositori Dataset IndQNER-EL dari GitHub

In [7]:
import os

# Kloning repositori resmi jika dijalankan di Google Colab
if not os.path.exists('IndQNER-EL-Beta'):
    !git clone https://github.com/fadhillahakbar885/IndQNER-EL-Beta.git
    %cd IndQNER-EL-Beta
else:
    %cd IndQNER-EL-Beta

!git pull origin main
!ls -la data/indqner_revised/

Cloning into 'IndQNER-EL-Beta'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 103 (delta 27), reused 92 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 1.96 MiB | 5.04 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/IndQNER-EL-Beta/IndQNER-EL-Beta
From https://github.com/fadhillahakbar885/IndQNER-EL-Beta
 * branch            main       -> FETCH_HEAD
Already up to date.
total 532
drwxr-xr-x 2 root root   4096 Oct 15 03:25 .
drwxr-xr-x 5 root root   4096 Oct 15 03:25 ..
-rw-r--r-- 1 root root  57955 Oct 15 03:25 dev.txt
-rw-r--r-- 1 root root  52038 Oct 15 03:25 test.txt
-rw-r--r-- 1 root root 418626 Oct 15 03:25 train.txt


## 3. Eksekusi Audit Kualitas Dataset (Pilar A)

In [8]:
# Verifikasi matematis Zero Deletion Policy dan keutuhan format BIO
!python src/05_audit_and_verification.py

MEMULAI AUDIT DAN VERIFIKASI INTEGRITAS DATASET

[VALIDASI DEV] MD5 Asli (34aa4e6c1828983e74e089034b79d4f7) == Revisi (34aa4e6c1828983e74e089034b79d4f7) -> 100% IDENTIK!
[VALIDASI TEST] MD5 Asli (b6039420734d81301975513a5a552cb6) == Revisi (b6039420734d81301975513a5a552cb6) -> 100% IDENTIK!

AUDIT INTEGRITAS SPLIT: TRAIN
Jumlah Baris Asli   : 51,570
Jumlah Baris Revisi : 51,570
  [PASSED] Jumlah baris identik 100%.
  [PASSED] 100% kata/token tidak mengalami perubahan sedikitpun.
  [PASSED] STRICT ZERO DELETION: 0 label ground-truth yang terhapus.
  [PASSED] STRICT PRESERVATION: 0 label entitas asli yang tertimpa label lain.
  [STATUS] Token berhasil diperkaya dari 'O' ke Label Entitas: 105
  [PASSED] Skema BIO valid 100%: Tidak ada tag 'I-' yang yatim/invalid.

Laporan Markdown lengkap berhasil dibuat di: /content/IndQNER-EL-Beta/IndQNER-EL-Beta/reports/stats_enrichment_summary.md
Log rincian audit disimpan di: /content/IndQNER-EL-Beta/IndQNER-EL-Beta/reports/audit_verification.log


## 4. Eksekusi Pelatihan & Evaluasi Komparatif IndoBERT (Pilar B)
> **Konfigurasi Parameter:** `epochs=10`, `batch_size=16`, `learning_rate=2e-5`, `max_length=256` (100% identik dengan README IndQNER)

In [9]:
# Melatih Model Baseline (Asli) dan Model Proposed (Revisi) sekaligus
!python src/train_eval_indobert.py --mode both --epochs 10 --batch_size 16 --learning_rate 2e-5 --max_length 256


MEMULAI EKSPERIMEN: BASELINE_INDQNER_ORIGINAL
Data Latih    : /content/IndQNER-EL-Beta/IndQNER-EL-Beta/IndQNER/datasets
Data Uji      : /content/IndQNER-EL-Beta/IndQNER-EL-Beta/IndQNER/datasets
Model Dasar   : indobenchmark/indobert-base-p1
Menggunakan Perangkat Komputasi: CUDA
Nama GPU: Tesla T4
[transformers] You passed `num_labels=27` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100% 197/197 [00:00<00:00, 13062.44it/s]
[transformers] BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your down

## 5. Menampilkan Tabel Hasil Komparasi untuk Naskah Skripsi

In [10]:
from IPython.display import Markdown, display

comparison_report = 'reports/evaluation_results/benchmark_comparison.md'
if os.path.exists(comparison_report):
    with open(comparison_report, 'r', encoding='utf-8') as f:
        content = f.read()
    display(Markdown(content))
else:
    print('Laporan belum tersedia. Pastikan sel pelatihan di atas sudah selesai dijalankan.')

Laporan belum tersedia. Pastikan sel pelatihan di atas sudah selesai dijalankan.
